# FunctionGemma 270M IT testing on Salesforce dataset 1000 samples

FunctionGemma is a lightweight, open model from Google, built as a foundation for creating your own specialized function calling models.

**Model Link:** [google/functiongemma](https://www.kaggle.com/models/google/functiongemma)

**Huggingface Salesforce Dataset:** [salesforce/xlam-function-calling-60k](https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k)

**Function calling models fine-tuning experiments:** https://github.com/silvermete0r/nano-slms-for-local-function-calling-assistance

**IMPORTANT NOTES:**

> FunctionGemma requires additional fine-tuning for specific tasks and, by default, produces outputs using a set of specialized formatting control tokens defined in the official documentation:
https://ai.google.dev/gemma/docs/functiongemma/formatting-and-best-practices

> Using AI-assisted tools (Claude / ChatGPT / Gemini), I attempted to build a converter (functiongemma -> json xlam-60k format) and tested it, but it does not always work reliably across all cases. So, it is important to emphasize that this is not a benchmark or a meaningful accuracy evaluation, since the model was not fine-tuned for this task and relies on its own specific tokens and formatting rules. As a result, the observed outputs do not reflect true model accuracy.

> Instead, this notebook is intended for exploratory analysis -> to better understand how FunctionGemma behaves in practice and to examine its interaction with function-calling formats, as well as to observe related performance (vram-usage, ttft, etc.) characteristics.

**Example (xlam-60k format):**

**Question:** `What’s the weather like in Aktobe today, and will there be rain this week?`

**Tools:** 
```json
[{"name": "get_weather_forecast", "description": "Retrieve the current weather and short-term forecast for a specific city.", "parameters": {"city": {"description": "The city to retrieve weather data for.", "type": "str"}, "forecast_days": {"description": "Number of days to include in the forecast.", "type": "int", "default": 7}}}]
```

**Answer:** 
```json
[{"name": "get_weather_forecast", "arguments": {"city": "Aktobe", "forecast_days": 7}}]
```

**References:**

This project uses data derived from the Salesforce xLAM Function Calling 60k dataset: https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k | Licensed under CC BY 4.0: https://creativecommons.org/licenses/by/4.0/

```
@article{liu2024apigen,
  title={APIGen: Automated Pipeline for Generating Verifiable and Diverse Function-Calling Datasets},
  author={Liu, Zuxin and Hoang, Thai and Zhang, Jianguo and Zhu, Ming and Lan, Tian and Kokane, Shirley and Tan, Juntao and Yao, Weiran and Liu, Zhiwei and Feng, Yihao and others},
  journal={arXiv preprint arXiv:2406.18518},
  year={2024}
}
```

## Install Dependencies

In [1]:
!pip install -q torch
!pip install -q transformers
!pip install -q codecarbon

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.5/380.5 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 93.4 MB/s eta 0:00:00


## Imports & Setup (+ Huggingface Token Setup)

In [2]:
import json, os, re, time
from datetime import datetime

import numpy as np
import psutil
import torch
from codecarbon import EmissionsTracker
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoProcessor, AutoModelForCausalLM

In [3]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')

login(HF_TOKEN)

In [4]:
import warnings
warnings.filterwarnings("ignore")

## Load Data

In [5]:
from datasets import load_dataset

RANDOM_STATE = 67 # for reproducibility

hf_dataset = load_dataset(
    "Salesforce/xlam-function-calling-60k",
    split="train",
    token=HF_TOKEN
)

def add_total_input_length(example):
    query_len = len(str(example["query"]))
    tools_len = len(str(example["tools"]))
    example["total_input_length"] = query_len + tools_len
    return example

hf_dataset = hf_dataset.map(add_total_input_length)

# keep only rows with total_input_length < 4096
filtered_dataset = hf_dataset.filter(
    lambda x: x["total_input_length"] < 4096
)

# Dataset ~ get: 10,000 samples
subset = filtered_dataset.shuffle(seed=RANDOM_STATE).select(range(10000))

# TEST SET ~ 10%
split_set = subset.train_test_split(test_size=1000, seed=RANDOM_STATE)
test_dataset = split_set['test']

print(f"Test Dataset: {len(test_dataset)} examples")

README.md: 0.00B [00:00, ?B/s]

xlam_function_calling_60k.json:   0%|          | 0.00/96.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Map:   0%|          | 0/60000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/60000 [00:00<?, ? examples/s]

Test Dataset: 1000 examples


## GPU (cuda) Info Preview + GPU Memory Utils

In [6]:
import torch

if torch.cuda.is_available():
    print("CUDA is available ✅")
    print("Device count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"\n--- GPU {i} ---")
        print("Name:", torch.cuda.get_device_name(i))
        print("Capability:", torch.cuda.get_device_capability(i))
        print("Total memory (GB):", torch.cuda.get_device_properties(i).total_memory / 1e9)
else:
    print("CUDA is NOT available ❌")

# GPU memory utils
def get_gpu_memory_mb(device=0):
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated(device) / 1024**2
    return 0.0

def get_vram_reserved_mb(device=0):
    if torch.cuda.is_available():
        return torch.cuda.memory_reserved(device) / 1024**2
    return 0.0

CUDA is available ✅
Device count: 2

--- GPU 0 ---
Name: Tesla T4
Capability: (7, 5)
Total memory (GB): 15.637086208

--- GPU 1 ---
Name: Tesla T4
Capability: (7, 5)
Total memory (GB): 15.637086208


## Load Model + Processor [FunctionGemma]

In [7]:
import kagglehub

model_path = kagglehub.model_download("google/functiongemma/transformers/functiongemma-270m-it")

processor = AutoProcessor.from_pretrained(model_path, device_map="cuda:0")
model = AutoModelForCausalLM.from_pretrained(model_path, dtype="auto", device_map="cuda:0")

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

In [8]:
model.eval()
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

Memory footprint: 0.54 GB


## FunctionGemma output -> JSON converter

In [9]:
"""
FunctionGemma → xLAM-style JSON converter
Spec: https://ai.google.dev/gemma/docs/functiongemma/formatting-and-best-practices
 
Output format: [{"name": str, "arguments": {str: Any}}, ...]
 
Key design decisions vs. the original converter:
  - No regex over the full body: the `}` anchor breaks on JSON-valued args.
    Instead we scan the raw text character-by-character inside the block body
    to correctly handle nested braces inside <escape>…<escape> spans.
  - Type coercion: unescaped tokens that look like int/float/bool/null are
    returned as native Python types, matching gold data produced by real schemas.
  - Function name pattern widened to [\\w.\\-]+ (dots and hyphens are valid).
  - Escape-tag stripping is done once, at value-extraction time only.
"""
 
import re
from typing import Any
 
# ── locate call blocks without anchoring on `}` ──────────────────────────────
 
_BLOCK_START_RE = re.compile(
    r"<start_function_call>\s*call:([\w.\-]+)\{",
)
_ESCAPE_OPEN = "<escape>"
_ESCAPE_LEN  = len(_ESCAPE_OPEN)   # 8  (open == close token)
 
 
def _extract_block_body(text: str, brace_open: int) -> tuple[str, int]:
    """
    Starting just after the opening `{` at index `brace_open`,
    scan forward tracking brace depth and escape spans.
    Returns (body_string, index_after_closing_brace).
 
    This is immune to `}` inside <escape>…<escape> spans.
    """
    depth   = 1
    in_esc  = False
    i       = brace_open
 
    while i < len(text) and depth > 0:
        # enter / leave escape span
        if text[i:i + _ESCAPE_LEN] == _ESCAPE_OPEN:
            in_esc = not in_esc
            i += _ESCAPE_LEN
            continue
 
        if not in_esc:
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                depth -= 1
 
        i += 1
 
    body = text[brace_open : i - 1]  # exclude the final `}`
    return body, i
 
 
# ── argument body parser ──────────────────────────────────────────────────────
 
def _coerce(value: str) -> Any:
    """
    Try to coerce a bare (unescaped) token to a Python native type.
    Escaped strings are always returned as-is.
    """
    low = value.lower()
    if low == "true":  return True
    if low == "false": return False
    if low in ("null", "none"): return None
    try:
        return int(value)
    except ValueError:
        pass
    try:
        return float(value)
    except ValueError:
        pass
    return value  # fallback: leave as string
 
 
def _parse_args(body: str) -> dict[str, Any]:
    """
    Parse `key:value, key:value, …` pairs from a call-block body.
 
    Rules (per spec):
      - String values are wrapped in <escape>…<escape>.
      - Bare (unescaped) values are scalars: numbers, booleans, null.
      - Commas inside <escape> spans are literal characters.
    """
    result: dict[str, Any] = {}
    i = 0
    n = len(body)
 
    while i < n:
        # skip leading whitespace / commas between pairs
        while i < n and body[i] in " \t\n\r,":
            i += 1
        if i >= n:
            break
 
        # ── read key (up to `:`) ─────────────────────────────────────────────
        key_start = i
        while i < n and body[i] != ":":
            i += 1
        key = body[key_start:i].strip()
        if not key or i >= n:
            break
        i += 1  # skip `:`
 
        # skip whitespace after colon
        while i < n and body[i] in " \t":
            i += 1
 
        # ── read value ───────────────────────────────────────────────────────
        if body[i:i + _ESCAPE_LEN] == _ESCAPE_OPEN:
            # escaped string: collect until the closing <escape>
            i += _ESCAPE_LEN
            val_start = i
            while i < n:
                if body[i:i + _ESCAPE_LEN] == _ESCAPE_OPEN:
                    break
                i += 1
            raw_val = body[val_start:i]
            i += _ESCAPE_LEN  # skip closing <escape>
            result[key] = raw_val  # already a string, no coercion
 
        else:
            # bare token: collect until next comma (outside any escape)
            # (nested braces can appear in responses but not in calls per spec)
            val_chars: list[str] = []
            while i < n and body[i] not in ",\n":
                val_chars.append(body[i])
                i += 1
            raw_val = "".join(val_chars).strip()
            result[key] = _coerce(raw_val)
 
    return result
 
 
# ── public API ────────────────────────────────────────────────────────────────
 
def functiongemma_response_to_json(raw: str) -> list[dict[str, Any]]:
    """
    Convert FunctionGemma model output to xLAM-style call list.
 
    Returns:
        [{"name": <fn_name>, "arguments": {<key>: <value>, ...}}, ...]
 
    Handles:
        - Single and parallel (multiple) function calls.
        - String values delimited by <escape>…<escape>.
        - Numeric / boolean / null bare values with type coercion.
        - Nested braces and commas inside escaped string values.
        - No-call (plain text) output → returns [].
    """
    results: list[dict[str, Any]] = []
 
    for m in _BLOCK_START_RE.finditer(raw):
        fn_name    = m.group(1)
        body_start = m.end()          # index right after the opening `{`
        body, _    = _extract_block_body(raw, body_start)
        args       = _parse_args(body)
        results.append({"name": fn_name, "arguments": args})
 
    return results


# ── tests ─────────────────────────────────────────────────────────────────────
import json

def check(label: str, raw: str, expected: list[dict]):
    got = functiongemma_response_to_json(raw)
    ok  = got == expected
    mark = "✅" if ok else "❌"
    print(f"{mark}  {label}")
    if not ok:
        print(f"   expected: {json.dumps(expected, ensure_ascii=False)}")
        print(f"   got:      {json.dumps(got,      ensure_ascii=False)}")

# 1. Basic string arg (from spec example)
check(
    "basic string arg",
    "<start_function_call>call:get_current_temperature"
    "{location:<escape>Almaty<escape>}"
    "<end_function_call>",
    [{"name": "get_current_temperature", "arguments": {"location": "Almaty"}}],
)

# 2. Multiple string args
check(
    "two string args",
    "<start_function_call>call:get_current_temperature"
    "{location:<escape>Paris, FR<escape>,unit:<escape>celsius<escape>}"
    "<end_function_call>",
    [{"name": "get_current_temperature",
      "arguments": {"location": "Paris, FR", "unit": "celsius"}}],
)

# 3. Numeric bare value (type coercion)
check(
    "numeric bare value",
    "<start_function_call>call:set_alarm"
    "{hour:7,minute:30}"
    "<end_function_call>",
    [{"name": "set_alarm", "arguments": {"hour": 7, "minute": 30}}],
)

# 4. Boolean and null coercion
check(
    "bool and null",
    "<start_function_call>call:toggle"
    "{enabled:true,value:null}"
    "<end_function_call>",
    [{"name": "toggle", "arguments": {"enabled": True, "value": None}}],
)

# 5. Comma inside escaped string value (THE key regression test)
check(
    "comma inside escape span",
    "<start_function_call>call:search"
    "{query:<escape>cats, dogs, and fish<escape>}"
    "<end_function_call>",
    [{"name": "search", "arguments": {"query": "cats, dogs, and fish"}}],
)

# 6. Closing brace `}` inside escaped string value (original regex breaks here)
check(
    "brace inside escape span",
    "<start_function_call>call:send_json"
    r'{payload:<escape>{"key": "val"}<escape>}'
    "<end_function_call>",
    [{"name": "send_json", "arguments": {"payload": '{"key": "val"}'}}],
)

# 7. Parallel calls
check(
    "parallel calls",
    "<start_function_call>call:get_weather"
    "{city:<escape>Tokyo<escape>}"
    "<end_function_call>"
    "<start_function_call>call:get_stock"
    "{ticker:<escape>GOOG<escape>}"
    "<end_function_call>",
    [
        {"name": "get_weather", "arguments": {"city": "Tokyo"}},
        {"name": "get_stock",   "arguments": {"ticker": "GOOG"}},
    ],
)

# 8. No function call (plain text) → empty list
check(
    "no call plain text",
    "I'm sorry, I don't know how to help with that.",
    [],
)

# 9. Mixed text + call (model preamble before the block)
check(
    "preamble text before call",
    "Sure! I'll look that up.\n"
    "<start_function_call>call:lookup{id:<escape>42<escape>}<end_function_call>",
    [{"name": "lookup", "arguments": {"id": "42"}}],
)

# 10. Float coercion
check(
    "float value",
    "<start_function_call>call:set_temp{degrees:98.6}<end_function_call>",
    [{"name": "set_temp", "arguments": {"degrees": 98.6}}],
)

✅  basic string arg
✅  two string args
✅  numeric bare value
✅  bool and null
✅  comma inside escape span
✅  brace inside escape span
✅  parallel calls
✅  no call plain text
✅  preamble text before call
✅  float value


In [10]:
idx = 7

answer = json.loads(test_dataset[idx]["answers"])

function_schemas = []
tools = json.loads(test_dataset[idx]["tools"])
for tool in tools:
    fg_tool = tool.copy()
    new_parameters = dict()
    new_parameters["type"] = "object"
    new_parameters["properties"] = fg_tool["parameters"] 
    new_parameters["required"] = list(fg_tool["parameters"].keys())
    fg_tool["parameters"] = new_parameters
    function_gemma_tool_body = {
        "type": "function",
        "function": fg_tool
    }
    function_schemas.append(function_gemma_tool_body)

print("query:\n", test_dataset[idx]["query"])
print("-" * 50)
print("tools:\n", function_schemas)
print("-" * 50)
print("answer:\n", answer)

query:
 I need the play-by-play data for the NHL game with the ID 9999999999. Is that possible?
--------------------------------------------------
tools:
 [{'type': 'function', 'function': {'name': 'get_play_by_play', 'description': 'Fetch the NHL game play-by-play data for a specified game.', 'parameters': {'type': 'object', 'properties': {'is_id': {'description': 'The unique identifier for the NHL game.', 'type': 'str', 'default': '401458986'}}, 'required': ['is_id']}}}]
--------------------------------------------------
answer:
 [{'name': 'get_play_by_play', 'arguments': {'is_id': '9999999999'}}]


In [11]:
message = [
    # ESSENTIAL SYSTEM PROMPT:
    # This line activates the model's function calling logic.
    {
        "role": "developer",
        "content": "You are a model that can do function calling with the following functions"
    },
    {
        "role": "user", 
        "content": test_dataset[idx]["query"]
    }
]

inputs = processor.apply_chat_template(message, tools=function_schemas, add_generation_prompt=True, return_dict=True, return_tensors="pt")

out = model.generate(**inputs.to(model.device), pad_token_id=processor.eos_token_id, max_new_tokens=1024)
output = processor.decode(out[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)

print("original functiongemma output:\n", output)
print("-" * 50)
fg_response = functiongemma_response_to_json(output)
print("corrected_json_output:\n", fg_response)

original functiongemma output:
 <start_function_call>call:get_play_by_play{is_id:<escape>9999999999<escape>}<end_function_call>
--------------------------------------------------
corrected_json_output:
 [{'name': 'get_play_by_play', 'arguments': {'is_id': '9999999999'}}]


In [12]:
assert answer == fg_response, f"FAILED: {answer} != {fg_response}"

## Run inference FunctionGemma

In [13]:
def run_inference_functiongemma(
    model_, processor_, rows: list[dict], max_print: int = 10
) -> list[dict]:
    all_results = []
    proc = psutil.Process(os.getpid())
    cnt_print = 0
    for row in tqdm(rows, desc="Samples"):
        tools_parsed = None
        try:
            tools_parsed = []
            raw_tools = json.loads(row["tools"])
            fg_tools = raw_tools.copy()
            if not isinstance(fg_tools, list):
                fg_tools = [fg_tools]
            for i in range(len(fg_tools)):
                if not isinstance(fg_tools[i], dict):
                    continue
                fg_tool = fg_tools[i].copy()
                new_parameters = dict()
                new_parameters["type"] = "object"
                new_parameters["properties"] = fg_tool["parameters"] 
                new_parameters["required"] = list(fg_tool["parameters"].keys())
                fg_tool["parameters"] = new_parameters
                function_gemma_tool_body = {
                    "type": "function",
                    "function": fg_tool
                }
                tools_parsed.append(function_gemma_tool_body)
        except Exception:
            pass
        
        message = [
            {
                "role":    "developer",
                "content": "You are a model that can do function calling with the following functions",
            },
            {
                "role":    "user",
                "content": row["query"],
            },
        ]
 
        inputs = processor_.apply_chat_template(
            message,
            tools=tools_parsed,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model_.device)
 
        input_len = len(inputs["input_ids"][0])
 
        ram_before  = proc.memory_info().rss / 1024**2
        vram_before = get_gpu_memory_mb()
        cpu_pct     = psutil.cpu_percent(interval=None)
 
        # TTFT: generate 1 token
        ttft_start = time.perf_counter()
        with torch.no_grad():
            model_.generate(
                **inputs,
                pad_token_id=processor_.eos_token_id,
                max_new_tokens=1,
            )
        ttft_s = time.perf_counter() - ttft_start
 
        # Full generation
        gen_start = time.perf_counter()
        with torch.no_grad():
            out = model_.generate(
                **inputs,
                pad_token_id=processor_.eos_token_id,
                max_new_tokens=1024,
            )
        total_time_s = time.perf_counter() - gen_start
 
        ram_after  = proc.memory_info().rss / 1024**2
        vram_after = get_gpu_memory_mb()
 
        # ── decode (strip prompt prefix, exactly as in docs) ─────────────────
        new_ids  = out[0][input_len:]
        raw_text = processor_.decode(new_ids, skip_special_tokens=True)
 
        tokens_gen = int(new_ids.shape[0])
 
        # ── convert FunctionGemma format → JSON string ───────────────────────
        calls        = functiongemma_response_to_json(raw_text)
        response_text = json.dumps(calls, ensure_ascii=False)

        if cnt_print < max_print:
            print("input_query:\n", row["query"])
            print("-" * 50)
            print("input_tools:\n", tools_parsed)
            print("-" * 50)
            print("output_raw_text:\n", raw_text)
            print("-" * 50)
            print("original_answer:\n", row["answers"])
            print("-" * 50)
            print("output_json:\n", response_text)
            print("\n" + "." * 50 + "\n\n")
            cnt_print += 1
            
        all_results.append({
            "response_text":    response_text,
            "raw_text":         raw_text,
            "total_time_s":     round(total_time_s, 4),
            "ttft_s":           round(ttft_s, 4),
            "tokens_generated": tokens_gen,
            "tokens_per_sec":   round(tokens_gen / total_time_s, 2) if total_time_s > 0 else 0.0,
            "vram_used_mb":     round(vram_after - vram_before, 2),
            "vram_peak_mb":     round(get_vram_reserved_mb(), 2),
            "ram_delta_mb":     round(ram_after - ram_before, 2),
            "cpu_percent":      round(cpu_pct, 1),
        })
 
    return all_results

## FunctionGemma Testing...

In [14]:
##### HELPERS #####
def safe_parse_json(value, fallback=None):
    if value is None: return fallback
    if isinstance(value, (dict, list)): return value
    if isinstance(value, str):
        try: return json.loads(value)
        except (json.JSONDecodeError, ValueError): return fallback
    return fallback

def safe_get_args(call: dict) -> dict:
    for key in ("arguments", "parameters", "args", "input"):
        if key in call:
            result = safe_parse_json(call[key], fallback={})
            if isinstance(result, dict): return result
    return {}

In [15]:
from collections import Counter

def benchmark(
    model_,
    processor_,
    test_ds,
    label:      str  = "FunctionGemma-270M-IT",
    save_json:  bool = True,
    json_path:  str  = "benchmark_results.json",
) -> dict:
 
    valid_rows = [
        r for r in test_ds
        if isinstance(r.get("query"), str) and isinstance(r.get("tools"), str)
    ]
    n_total = len(valid_rows)
 
    tracker = EmissionsTracker(
        project_name=f"benchmark_{label.replace(' ', '_')}",
        log_level="error",
        save_to_file=False,
    )
    tracker.start()
    wall_start = time.time()
 
    all_results = run_inference_functiongemma(
        model_, processor_, valid_rows
    )
 
    wall_total = time.time() - wall_start
    co2_kg     = tracker.stop()
 
    # ──── scoring ────
    json_valid = name_match = args_exact = args_keys_match = 0
    n = 0
 
    perf_keys  = ["total_time_s", "ttft_s", "tokens_per_sec",
                  "vram_used_mb", "vram_peak_mb", "ram_delta_mb",
                  "cpu_percent",  "tokens_generated"]
    perf_accum = {k: [] for k in perf_keys}

    # fixed bugs: https://github.com/silvermete0r/nano-slms-for-local-function-calling-assistance/issues/2
    for row, res in zip(valid_rows, all_results):
        for k in perf_keys:
            perf_accum[k].append(res[k])

        pred = safe_parse_json(res["response_text"], fallback=None)
        gold = safe_parse_json(row.get("answers"), fallback=None)

        if not isinstance(gold, list):
            continue
        gold_calls = [c for c in gold if isinstance(c, dict) and c.get("name")]
        if not gold_calls:
            continue
        n += 1

        if not isinstance(pred, list):
            continue
        pred_calls = [c for c in pred if isinstance(c, dict)]
        json_valid += 1

        # ── name_match ──
        gold_names = Counter(c["name"] for c in gold_calls)
        pred_names = Counter(c.get("name") for c in pred_calls)
        if gold_names == pred_names:
            name_match += 1

        # ── align calls by name for args scoring ──
        # Group both gold and pred by name, then match call-by-call within each group.
        # If a name appears multiple times, sort args by their string repr for
        # a stable, order-independent pairing.
        def group_by_name(calls):
            groups = {}
            for c in calls:
                name = c.get("name")
                if name:
                    groups.setdefault(name, []).append(safe_get_args(c))
            # stable sort within each name bucket so pairing is deterministic
            for name in groups:
                groups[name].sort(key=lambda a: str(sorted(a.items())))
            return groups

        gold_grouped = group_by_name(gold_calls)
        pred_grouped = group_by_name(pred_calls)

        all_keys_match = True
        all_exact      = True

        for name, g_args_list in gold_grouped.items():
            p_args_list = pred_grouped.get(name, [])
            for idx, g_args in enumerate(g_args_list):
                p_args = p_args_list[idx] if idx < len(p_args_list) else {}
                if g_args.keys() != p_args.keys():
                    all_keys_match = False
                if g_args != p_args:
                    all_exact = False

        if all_keys_match:
            args_keys_match += 1
        if all_exact:
            args_exact += 1
 
    if n == 0:
        print(f"[{label}] No scorable rows found.")
        return {}
 
    def avg(lst): return round(float(np.mean(lst)),       4) if lst else 0.0
    def p95(lst): return round(float(np.percentile(lst, 95)), 4) if lst else 0.0
 
    results = {
        "label":         label,
        "timestamp":     datetime.utcnow().isoformat() + "Z",
        "total_samples": n,
        "batch_size": 1,
        "accuracy": {
            "json_valid_pct":      round(100 * json_valid      / n, 1),
            "name_match_pct":      round(100 * name_match      / n, 1),
            "args_keys_match_pct": round(100 * args_keys_match / n, 1),
            "args_exact_pct":      round(100 * args_exact      / n, 1),
        },
        "performance": {
            "wall_total_s":           round(wall_total, 2),
            "avg_latency_s":          avg(perf_accum["total_time_s"]),
            "p95_latency_s":          p95(perf_accum["total_time_s"]),
            "avg_ttft_s":             avg(perf_accum["ttft_s"]),
            "avg_tokens_per_sec":     avg(perf_accum["tokens_per_sec"]),
            "avg_tokens_generated":   avg(perf_accum["tokens_generated"]),
            "avg_vram_delta_mb":      avg(perf_accum["vram_used_mb"]),
            "peak_vram_reserved_mb":  avg(perf_accum["vram_peak_mb"]),
            "avg_ram_delta_mb":       avg(perf_accum["ram_delta_mb"]),
            "avg_cpu_percent":        avg(perf_accum["cpu_percent"]),
            "throughput_samples_per_sec": round(n / wall_total, 2),
        },
        "co2": {
            "emissions_kg": round(co2_kg, 6) if co2_kg else None,
            "emissions_g":  round(co2_kg * 1000, 4) if co2_kg else None,
        },
    }
 
    print(f"\n{'='*52}")
    print(f"  {label}")
    print(f"{'='*52}")
    print("  ACCURACY")
    for k, v in results["accuracy"].items():
        print(f"    {k:<28}: {v} %")
    print("  PERFORMANCE")
    for k, v in results["performance"].items():
        print(f"    {k:<28}: {v}")
    if co2_kg:
        print(f"  CO₂  {results['co2']['emissions_g']} g CO₂eq")
    print(f"{'='*52}\n")
 
    if save_json:
        existing = []
        if os.path.exists(json_path):
            with open(json_path) as f:
                try:    existing = json.load(f)
                except: existing = []
        existing.append(results)
        with open(json_path, "w") as f:
            json.dump(existing, f, indent=2)
        print(f"  ✅  Results appended → {json_path}")
 
    return results

In [16]:
JSON_PATH = "benchmark_results.json"

results = benchmark(
    model,
    processor,
    test_dataset,
    label="FunctionGemma-270M-IT (base)",
    json_path=JSON_PATH
)

[codecarbon WARNING @ 15:37:25] Multiple instances of codecarbon are allowed to run at the same time.
Samples:   0%|          | 1/1000 [00:01<27:49,  1.67s/it]

input_query:
 What is the area of a quadrilateral with vertices at (0, 0), (2, 0), (3, 4), and (1, 4)?
--------------------------------------------------
input_tools:
 [{'type': 'function', 'function': {'name': 'is_valid_sudoku', 'description': 'Checks if a 9x9 Sudoku board is valid.', 'parameters': {'type': 'object', 'properties': {'board': {'description': 'The Sudoku board represented as a 2D list of strings.', 'type': 'List[List[str]]'}}, 'required': ['board']}}}, {'type': 'function', 'function': {'name': 'polygon_area_shoelace', 'description': 'Calculates the area of a polygon using the shoelace formula.', 'parameters': {'type': 'object', 'properties': {'vertices': {'description': 'A list of polygon vertices represented as tuples (x, y).', 'type': 'List[Tuple[float, float]]'}}, 'required': ['vertices']}}}]
--------------------------------------------------
output_raw_text:
 <start_function_call>call:polygon_area_shoelace{vertices:[(0, 0), (2, 0), (3, 4), (1, 4))]}<end_function_call

Samples:   0%|          | 2/1000 [00:03<25:46,  1.55s/it]

input_query:
 What are the current trading signals for Bitcoin on Binance in the SPOT market?
--------------------------------------------------
input_tools:
 [{'type': 'function', 'function': {'name': 'get_signals', 'description': 'Fetches trading signals and trends for a specified cryptocurrency pair from the given exchange and market type.', 'parameters': {'type': 'object', 'properties': {'coin': {'description': 'The cryptocurrency pair to get signals for (e.g., BTC, ETH, TRX).', 'type': 'str', 'default': 'BTC'}, 'exchange': {'description': 'The exchange to get signals from (e.g., Binance, Bybit, Huobi, Kucoin, Coinex, MXC, Gate).', 'type': 'str', 'default': 'Bybit'}, 'market_type': {'description': 'The market type to get signals for (e.g., SPOT, FUTURES).', 'type': 'str', 'default': 'SPOT'}}, 'required': ['coin', 'exchange', 'market_type']}}}]
--------------------------------------------------
output_raw_text:
 <start_function_call>call:get_signals{coin:<escape>BTC<escape>,exchange

Samples:   0%|          | 3/1000 [00:05<31:16,  1.88s/it]

...

input_query:
 Please provide a password for my new social media account, 11 characters, no special characters.
--------------------------------------------------
input_tools:
 [{'type': 'function', 'function': {'name': 'generate_password', 'description': 'Generates a random password of specified length and character types.', 'parameters': {'type': 'object', 'properties': {'length': {'description': 'The length of the password. Defaults to 12.', 'type': 'int, optional', 'default': 12}, 'include_special': {'description': 'Whether to include special characters in the password. Defaults to True.', 'type': 'bool, optional', 'default': True}}, 'required': ['length', 'include_special']}}}]
--------------------------------------------------
output_raw_text:
 <start_function_call>call:generate_password{include_special:false,length:11}<end_function_call>
--------------------------------------------------
original_answer:
 [{"name": "generate_password", "arguments": {"length": 11, "include_special

Samples:   1%|          | 10/1000 [00:15<21:42,  1.32s/it]

input_query:
 Fetch all responses for the question with the ID '67890' and display them.
--------------------------------------------------
input_tools:
 [{'type': 'function', 'function': {'name': 'latest_rates', 'description': 'Fetches the latest real-time exchange rates for given symbols based on a specified base currency.', 'parameters': {'type': 'object', 'properties': {'symbols': {'description': 'Comma-separated list of currency or commodity codes to retrieve rates for.', 'type': 'str', 'default': 'BRENTOIL'}, 'base': {'description': 'Three-letter currency or commodity code to use as the base currency.', 'type': 'str', 'default': 'USD'}}, 'required': ['symbols', 'base']}}}, {'type': 'function', 'function': {'name': 'search_countries_by_idd', 'description': 'Searches for countries using their International Direct Dialing (IDD) number.', 'parameters': {'type': 'object', 'properties': {'idd': {'description': "International Direct Dialing number, starting with '+'.", 'type': 'str', 'd

Samples: 100%|██████████| 1000/1000 [33:37<00:00,  2.02s/it]


  FunctionGemma-270M-IT (base)
  ACCURACY
    json_valid_pct              : 100.0 %
    name_match_pct              : 93.8 %
    args_keys_match_pct         : 71.7 %
    args_exact_pct              : 50.4 %
  PERFORMANCE
    wall_total_s                : 2017.01
    avg_latency_s               : 1.9485
    p95_latency_s               : 4.0797
    avg_ttft_s                  : 0.0645
    avg_tokens_per_sec          : 25.1087
    avg_tokens_generated        : 48.859
    avg_vram_delta_mb           : 0.0017
    peak_vram_reserved_mb       : 641.082
    avg_ram_delta_mb            : 0.0043
    avg_cpu_percent             : 27.7495
    throughput_samples_per_sec  : 0.5
  CO₂  14.3641 g CO₂eq

  ✅  Results appended → benchmark_results.json
